# 공통 되는 것

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
building_info = pd.read_csv('building_info.csv')
#train = pd.read_csv('data/train.csv')
#test = pd.read_csv('data/test.csv')
#building_info = pd.read_csv('data/building_info.csv')
# 2. '건물번호'를 기준으로 train과 test 데이터에 건물 정보 병합
train = pd.merge(train, building_info, on='건물번호', how='left')
test = pd.merge(test, building_info, on='건물번호', how='left')

In [3]:
for df in [train, test]:
        df['일시'] = pd.to_datetime(df['일시'])
        df['month'] = df['일시'].dt.month
        df['hour'] = df['일시'].dt.hour
        df['dayofweek'] = df['일시'].dt.dayofweek # day 없음
        # 시간의 주기성 (Sin/Cos)
        df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24)

In [4]:
# train 컬럼
#train_cols_to_fill0 = ['강수량(mm)', '일조(hr)', '일사(MJ/m2)']
#for col in train_cols_to_fill0:
#    train[col] = train[col].fillna(0)

# test 컬럼 (강수량만 존재)
#test_cols_to_fill0 = ['강수량(mm)']
#for col in test_cols_to_fill0:
#    test[col] = test[col].fillna(0)

In [5]:
cols_to_interp = ['기온(C)', '풍속(m/s)', '습도(%)']

train[cols_to_interp] = train[cols_to_interp].interpolate(method='linear')

# 위가 a, 아래가 b = a or b  둘 중에 1개 골라야 함   
# 아래는 [train, test] 둘다 *test에 강수량 있음
interp_cols = ['기온(C)', '풍속(m/s)', '습도(%)']

for col in interp_cols:

    if col in df.columns:

        df[col] = df[col].interpolate(method='linear', limit_direction='both')

결측치 <br>
    강수량, 일사,일조량은 = 0 <br>
         풍속 습도 기온 = 중간값 <br>
-> 강수량은 train, test 둘다 있음 <br>
-> 풍속 습도 기온 = 중간값이라고 되어 있는데 선형보간 안 쓰는지 의문. <br>
   만약 선형보간 쓴다고 가정하면 윗 셀에서 a, b 둘다 처리기법이 다름 <br>
-> 선형보간을 쓰지 않고, 그대로 중간값 쓰면 되는지 모르는 상태 (?) <br>

시간변수 <br>
    시간단위는 주기형 <br> 
    주말과 공휴일 변수 추가 <br>
    month, day, hour, day of week. 로 나누기 <br>

임시 휴무(할인마트 등)로 추측되는 데이터는 제거 (drop) <br> 

불쾌지수, 체감온도, 평균기온, 최대기온 <br>
-> 불쾌지수랑 체감온도 부분에서 둘다 처리기법이 다름 <br>

---------------------------------------------------------------------------------------
-건물 별- (권태은 황기수)
건물 성향, 유형이 어떤지 학습시켜서 분류시키기


-건물 유형별- (김재원 김성일)
건물 유형 별로 나누기. + 건물기타에 들어간 부분을 3가지로 나눠서 보기
각 건물 유형 별로 히트맵 시각화